## Project NLP and Deep Learning

### 1. Project proposal presentation

In the presentation, you have 5 minutes to present your research proposal. During the presentation, you should explain:

* What is the topic of your project, what is the current state of this topic/task/setup
* What is the new part of your project
* What is the research question of your project

We have proposed a number of topics in the slides which can be found on LearnIt, you can either pick one of these or come up with your own. If you pick your own, we suggest to get a pre-approval with Rob van der Goot.

**Deadline for uploading slides: day before the presentation (23:59)**  (pdf only, they will be put into one long pdf for a smooth presentation)

### 2. Baseline
To get your project started, you start with implementing a baseline model. Ideally, this is going to be the main baseline that you are going to compare to in your paper. Note that this baseline should be more advanced than just predicting the majority class (O).

We will use EWT portion of the [Universal NER project](http://www.universalner.org/), which we provide with this notebook for convenience. You can use the train data (`en_ewt-ud-train.iob2`) and dev data(`en_ewt-ud-dev.iob2`) to build your baseline, then upload your prediction on the test data (`en_ewt-ud-test.iob2`).

It is important to upload your predictions in same format as the training and dev files, so that the `span_f1.py` script can be used.

Note that you do not have to implement your baseline from scratch, you can use for example the code from the RNN or BERT assignments as a starting point.

**Deadline: 20-03 on LearnIt (14:00)**

In [2]:
import torch
import random
import numpy as np
from datasets import load_dataset, Dataset, DatasetDict
from transformers import (AutoTokenizer, AutoModelForTokenClassification, DataCollatorForTokenClassification, Trainer, TrainingArguments, set_seed, AutoConfig)
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

# Set seed for reproducibility
set_seed(42)

# Define constants
MODEL_NAME = "bert-base-uncased"
BATCH_SIZE = 8
EPOCHS = 3
LEARNING_RATE = 2e-5

# Load tokenizer
# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Load dataset using load_dataset
data_files = {
    "train": "en_ewt-ud-train.iob2",
    "validation": "en_ewt-ud-dev.iob2",
    "test": "en_ewt-ud-test-masked.iob2"
}

# Function to parse IOB2 dataset and extract label list
def parse_iob2(file_path):
    sentences, labels = [], []
    words, tags = [], []
    unique_labels = set()
    
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):  # Skip metadata lines
                if words:
                    sentences.append(words)
                    labels.append(tags)
                    words, tags = [], []
                continue
            parts = line.split()  # Split by whitespace
            if len(parts) >= 2:
                words.append(parts[1])  # First column: token
                tags.append(parts[2])  # Second column: label
                unique_labels.add(parts[2])
        if words:
            sentences.append(words)
            labels.append(tags)
    
    return {"tokens": sentences, "ner_tags": labels}, sorted(unique_labels)


dataset_dict = {}
all_labels = set()
for split, file in data_files.items():
    parsed_data, labels = parse_iob2(file)
    dataset_dict[split] = parsed_data
    all_labels.update(labels)

# Convert label list to mapping
label_list = sorted(all_labels)  # Ensure consistent order
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for label, i in label2id.items()}

# Convert labels to integers
def convert_labels(dataset):
    dataset["ner_tags"] = [[label2id[tag] for tag in tags] for tags in dataset["ner_tags"]]
    return dataset

dataset_dict = {split: convert_labels(dataset) for split, dataset in dataset_dict.items()}
raw_datasets = DatasetDict({
    split: Dataset.from_dict(dataset_dict[split]) for split in dataset_dict
})

# Load model with proper config
config = AutoConfig.from_pretrained(MODEL_NAME, num_labels=len(label_list), id2label=id2label, label2id=label2id)
model = AutoModelForTokenClassification.from_pretrained(MODEL_NAME, config=config)




c:\Users\tettret\Downloads\Anaconda\envs\week6_assignment_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# Tokenization function with label alignment
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        max_length=128, 
        padding="max_length",  # Ensures consistent length
        truncation=True, 
        is_split_into_words=True
    )

    all_labels = []
    for batch_index, labels in enumerate(examples["ner_tags"]):  # Labels are already integers
        word_ids = tokenized_inputs.word_ids(batch_index=batch_index)
        label_ids = []
        prev_word_id = None

        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)  # Special tokens get -100
            elif word_id == prev_word_id:
                label_ids.append(-100)  # Subword tokens get -100
                ...
            else:
                label_ids.append(labels[word_id])  # Use the mapped label directly

            prev_word_id = word_id

        all_labels.append(label_ids)

    tokenized_inputs["labels"] = all_labels
    return tokenized_inputs


# Apply tokenization
tokenized_datasets = raw_datasets.map(tokenize_and_align_labels, batched=True)

Map:   0%|          | 0/12543 [00:00<?, ? examples/s]

Map: 100%|██████████| 2077/2077 [00:00<00:00, 8634.10 examples/s]


In [65]:
label2id

{'B-LOC': 0,
 'B-ORG': 1,
 'B-PER': 2,
 'I-LOC': 3,
 'I-ORG': 4,
 'I-PER': 5,
 'O': 6}

In [30]:
# Map the tokenize_and_align_labels function to the raw datasets

train_dataset = tokenized_datasets["train"]
eval_dataset = tokenized_datasets["validation"]

# Inspect a few training samples after tokenization
for index in random.sample(range(len(train_dataset)), 3):
    print(f"Sample {index} of the training set: {train_dataset[index]}")

Sample 10476 of the training set: {'tokens': ['Overall', ',', 'I', 'was', 'very', 'happy', 'with', 'the', 'customer', 'service', 'and', 'my', 'purchase', '.'], 'ner_tags': [6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6], 'input_ids': [101, 3452, 1010, 1045, 2001, 2200, 3407, 2007, 1996, 8013, 2326, 1998, 2026, 5309, 1012, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [22]:
# Initialize the model with AutoModelForTokenClassification
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    config=config
)

# Create a data collator
data_collator = DataCollatorForTokenClassification(tokenizer)

Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
from torch.utils.data import DataLoader
import torch
from tqdm.auto import tqdm
from transformers import DataCollatorForTokenClassification

# Move model to device (CPU/GPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Proper data collator for token classification (handles padding)
data_collator = DataCollatorForTokenClassification(tokenizer, padding=True)
# Ensure proper batching and padding
data_collator = DataCollatorForTokenClassification(tokenizer, padding=True)

# Filter out unwanted columns
columns_to_remove = ["tokens", "ner_tags", "token_type_ids"]

# Remove extra columns
#tokenized_datasets = tokenized_datasets.remove_columns(columns_to_remove)
# Take a subset of the dataset (e.g., first 1000 samples)
small_train_dataset = tokenized_datasets["train"]
small_eval_dataset = tokenized_datasets["validation"].select(range(200))  # Smaller eval set


train_dataloader = DataLoader(
    small_train_dataset, shuffle=True, collate_fn=data_collator, batch_size=BATCH_SIZE
)
eval_dataloader = DataLoader(
    small_eval_dataset, collate_fn=data_collator, batch_size=BATCH_SIZE
)

# Define optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

# Training loop
model.train()
for epoch in range(EPOCHS):
    total_loss = 0
    pbar = tqdm(train_dataloader, desc=f"Training Epoch {epoch+1}")

    for batch in pbar:
        batch = {key: val.to(device) for key, val in batch.items() if isinstance(val, torch.Tensor)}
        batch["labels"] = batch["labels"].long()  # Ensure labels are LongTensor
        
        optimizer.zero_grad()  # Zero gradients before forward pass
        
        # Forward pass
        outputs = model(**batch)
        loss = outputs.loss

        # Backward pass
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        pbar.set_postfix(loss=loss.item())

    avg_loss = total_loss / len(train_dataloader)
    print(f"Epoch {epoch+1}/{EPOCHS}, Average Loss: {avg_loss:.4f}")

# Save model after training
model.save_pretrained("./bert_ner_manual")
tokenizer.save_pretrained("./bert_ner_manual")

# Evaluation loop
model.eval()
total_eval_loss = 0

for batch in tqdm(eval_dataloader, desc="Evaluating"):
    batch = {key: val.to(device) for key, val in batch.items() if isinstance(val, torch.Tensor)}
    batch["labels"] = batch["labels"].long()  # Ensure labels are LongTensor
    
    with torch.no_grad():
        outputs = model(**batch)
    
    total_eval_loss += outputs.loss.item()

avg_eval_loss = total_eval_loss / len(eval_dataloader)
print(f"Validation Loss: {avg_eval_loss:.4f}")


Training Epoch 1: 100%|██████████| 125/125 [13:27<00:00,  6.46s/it, loss=0.0103] 


Epoch 1/3, Average Loss: 0.1050


Training Epoch 2: 100%|██████████| 125/125 [11:10<00:00,  5.36s/it, loss=0.0997] 


Epoch 2/3, Average Loss: 0.0579


Training Epoch 3: 100%|██████████| 125/125 [09:58<00:00,  4.79s/it, loss=0.168]   


Epoch 3/3, Average Loss: 0.0363


Evaluating: 100%|██████████| 25/25 [00:33<00:00,  1.35s/it]

Validation Loss: 0.1270


In [ ]:
import evaluate
data_collator = DataCollatorForTokenClassification(tokenizer)
model = AutoModelForTokenClassification.from_pretrained(MODEL_NAME, config=config)

small_eval_dataset = tokenized_datasets["validation"].select(range(200))  # Smaller eval set
# Define training arguments
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=6,
    weight_decay=0.001,
)

# Define trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator
)

# Train model
trainer.train()

# Evaluate model
metric = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)
    return metric.compute(predictions=predictions, references=labels)

trainer.evaluate(eval_dataset=tokenized_datasets["validation"], metric_key_prefix="eval")

# Generate predictions on test set
#def predict_and_save(test_dataset, output_file):
#    predictions = trainer.predict(test_dataset).predictions
#    predictions = np.argmax(predictions, axis=2)
#    with open(output_file, "w") as f:
#        for pred in predictions:
#            f.write(" ".join(map(str, pred)) + "\n")
#
#predict_and_save(tokenized_datasets["test"], "predictions.iob2")


In [ ]:
# **✅ Save the trained model and tokenizer**
SAVE_PATH = "./trained_model_2"
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f"✅ Model and tokenizer saved to {SAVE_PATH}")

✅ Model and tokenizer saved to ./trained_model


In [ ]:
from transformers import AutoModelForTokenClassification
import torch
import json
from transformers import AutoModelForTokenClassification, Trainer, TrainingArguments
import torch

data_collator = DataCollatorForTokenClassification(tokenizer)

# Load trained model
model = AutoModelForTokenClassification.from_pretrained("./trained_model")
model.eval()

# Initialize the Trainer
# Define training arguments (you can customize this)
training_args = TrainingArguments(
    output_dir="./results",  # where to store the results
    per_device_eval_batch_size=BATCH_SIZE,  # batch size during evaluation
    evaluation_strategy="epoch",  # how often to evaluate
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator
)

# Run inference on the test dataset
predictions = trainer.predict(tokenized_datasets["test"])
#predictions = trainer.predict(tokenized_datasets["validation"])

# Extract label predictions
pred_labels = torch.argmax(torch.tensor(predictions.predictions), dim=2).numpy()

# Convert predicted IDs to text labels
final_predictions = [
    [id2label[p] for p in sentence] for sentence in pred_labels
]



C:\Users\tettret\AppData\Local\Temp\ipykernel_30184\660040146.py:20: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
# Convert predictions to IOB format
iob_predictions = []

for sent_idx, final_pred in enumerate(final_predictions):
    tokens = tokenized_datasets["test"]["tokens"][sent_idx]  # Tokens from your dataset
    tokens_ids = tokenized_datasets["test"]["input_ids"][sent_idx]  # Tokens from your dataset
    #tokens = tokenized_datasets["validation"]["tokens"][sent_idx]  # Tokens from your dataset
    #tokens_ids = tokenized_datasets["validation"]["input_ids"][sent_idx]  # Tokens from your dataset
    sentence_iob = []
    pred_label_cleaned=final_pred[1:len(tokens)+1]
    for token_idx, token in enumerate(tokens):
        pred_label = pred_label_cleaned[token_idx]  # Get the label from final_predictions
        sentence_iob.append(f"{token_idx+1}\t{tokens[token_idx]}\t{pred_label}")
    
    iob_predictions.append(sentence_iob)

# Save predictions to a file in IOB format
with open("ner_predictions_dev.iob", "w") as f:
    for sentence in iob_predictions:
        for line in sentence:
            f.write(f"{line}\n")
        f.write("\n")  # Separate sentences with a blank line

Make predictions

In [ ]:
import torch
import numpy as np

# Define output file
output_file = "predictions.iob2"

# Assume `tokenized_datasets["test"]` contains tokenized input sentences
# Assume `label_map` exists to map model output indices to IOB2 labels
# Take first 200 samples
small_test_dataset = tokenized_datasets["test"].select(range(200))  
# Extract original tokens from test dataset
original_tokens = [example["tokens"] for example in small_test_dataset]
test_dataloader = DataLoader(
    small_test_dataset, collate_fn=data_collator, batch_size=BATCH_SIZE
)

# Ensure model is in eval mode
model.eval()
predictions = []

# Run inference
for batch in tqdm(test_dataloader, desc="Predicting"):
    batch = {key: val.to(device) for key, val in batch.items() if isinstance(val, torch.Tensor)}
    
    with torch.no_grad():
        outputs = model(**batch)

    # Convert logits to label indices
    preds = torch.argmax(outputs.logits, dim=-1).cpu().numpy()

    # Convert predictions to label format
    for pred in preds:
        predictions.append([id2label[label] for label in pred])  # Convert indices to labels

# Save predictions in IOB2 format
with open(output_file, "w", encoding="utf-8") as f:
    for tokens, preds in zip(original_tokens, predictions):
        for token, label in zip(tokens, preds):
            f.write(f"{token} {label}\n")
        f.write("\n")  # Add blank line to separate sentences

print(f"✅ Predictions saved successfully in IOB2 format: {output_file}")


In [ ]:
# Run inference
import json


model.eval()
predictions = []

for batch in tqdm(test_dataloader, desc="Predicting"):
    batch = {key: val.to(device) for key, val in batch.items() if isinstance(val, torch.Tensor)}
    
    with torch.no_grad():
        outputs = model(**batch)
    
    # Convert logits to label indices and ensure they are Python lists
    preds = torch.argmax(outputs.logits, dim=-1).cpu().numpy().tolist()  # Convert to list
    predictions.extend(preds)

# Save predictions as a JSON file
predictions_file = "predictions.json"
with open(predictions_file, "w") as f:
    json.dump(predictions, f, indent=4)  # Add indentation for readability

print(f"✅ Predictions saved successfully to {predictions_file}!")

Predicting: 100%|██████████| 25/25 [00:30<00:00,  1.22s/it]

✅ Predictions saved successfully to predictions.json!


In [ ]:
# Tokenization function
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(examples["tokens"], padding=True, truncation=True, is_split_into_words=True)
    return tokenized_inputs

# Tokenize datasets
tokenized_datasets = raw_datasets.map(tokenize_and_align_labels, batched=True)

data_collator = DataCollatorForTokenClassification(tokenizer)


# Define training arguments
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
)

# Define trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator
)

# Train model
trainer.train()

# Evaluate model
metric = load_metric("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)
    return metric.compute(predictions=predictions, references=labels)

trainer.evaluate(eval_dataset=tokenized_datasets["validation"], metric_key_prefix="eval")

# Generate predictions on test set
def predict_and_save(test_dataset, output_file):
    predictions = trainer.predict(test_dataset).predictions
    predictions = np.argmax(predictions, axis=2)
    with open(output_file, "w") as f:
        for pred in predictions:
            f.write(" ".join(map(str, pred)) + "\n")

predict_and_save(tokenized_datasets["test"], "predictions.iob2")

In [ ]:
# Load the tokenizer and model config
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
config = AutoConfig.from_pretrained(model_name, num_labels=len(label_list))

def tokenize_and_align_labels(examples):
    """
    For each example, tokenize the list of tokens and align the original labels 
    to the resulting subwords. Tokens can be split into multiple subwords, so we mark 
    the "extra" subwords with -100 to ignore them in the loss.
    """

    # 1) Tokenize
    # 'is_split_into_words=True' tells the tokenizer each item in the list is already a separate word/token.
    tokenized_inputs = tokenizer(
        examples[text_column_name],  # e.g., examples["tokens"]
        max_length=128,             
        padding=False,              
        truncation=True, 
        is_split_into_words=True
    )

    # 2) Prepare a new "labels" list aligned to the subword tokens
    all_labels = []
    
    # examples[label_column_name] might look like: [0, 0, 1, 2, ...] for each token
    for batch_index, labels in enumerate(examples[label_column_name]):
        # 'word_ids()' returns a list the same length as the subword-tokens,
        # each entry telling you which 'word' or token it came from
        word_ids = tokenized_inputs.word_ids(batch_index=batch_index)

        label_ids = []
        prev_word_id = None
        
        for word_id in word_ids:
            if word_id is None:
                # e.g. special tokens or padding
                label_ids.append(-100)
            elif word_id == prev_word_id:
                # subword token of the same word => ignore 
                label_ids.append(-100)
            else:
                # new subword, so use the label for the original token
                label_ids.append(labels[word_id])
            
            prev_word_id = word_id
        
        all_labels.append(label_ids)

    # 3) Attach the new "labels" to our tokenized inputs
    tokenized_inputs["labels"] = all_labels

    # 4) Return the updated dictionary
    return tokenized_inputs


In [ ]:
# Initialize the model with AutoModelForTokenClassification
model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    config=config
)

# Create a data collator
data_collator = DataCollatorForTokenClassification(tokenizer)


In [ ]:
# Identify columns in raw_datasets['train'].features
text_column_name = "tokens"
label_column_name = "ner_tags"

In [ ]:
# Load the dataset using load_dataset
dataset_name = "conll2003"
raw_datasets = load_dataset(dataset_name,trust_remote_code=True)

# Inspect the dataset structure
raw_datasets


### 3. Project proposal

The written proposal should consist of maximum one page in [ACL-format](https://github.com/acl-org/acl-style-files) (The bibliography does not count for the word limit). In here, you should explain the last three points from the list above and place your project in a larger context (previous work).

Make sure your proposal is:
* Novel to some extent
* Doable within the time-frame

*hint* The [ACL Anthology](https://aclanthology.org/) contains almost all peer-reviewed NLP papers.

**Deadline: 03-04 on LearnIt (14:00)**

### 4. Final project
The final project has a maximum size of 5 pages (excluding bibliography and appendix), using the [ACL style files](https://github.com/acl-org/acl-style-files)

Besides the main paper (discussed in class), you have to include:
* Group contributions. State who was responsible for which part of the project. Here you may state if there
were any serious unequal workloads among group members. This should be put in the appendix.
* A report on usage of chatbots. We follow: https://2023.aclweb.org/blog/ACL-2023-policy/
   * Add a section in appendix if you made use of a chatbot (since we do not use a Responsible NLP Checklist)
   * Include each stage on the ACL policy, and indicate to what extent you used a chatbot
   * Use with care!, you are responsible for the project and plagiarism, correctness etc.

You can also put additional results and details in the appendix. However, the paper itself should be standalone, and understandable without consulting the appendix.

Furthermore, the code should be available on www.github.itu.dk (with a link in a footnote at the end of the abstract) , it should include a README with instructions on how to reproduce your results.

**Deadline: 23-05 on LearnIt** Please check the checklist below before uploading!

Optionally, you can upload a draft a week before **16-05 (before 09:00)** for an extra round of feedback

## Analysis

Analysis is essential for the interpretation of your results. In this section we will shortly describe some different types of analysis. We strongly suggest to use at least one of these:

* **Ablation study**: Leave out a certain part of the model, to study its effects. For example, disable the tokenizer, remove a certain (group of) feature(s), or disable the stop-word removal. If the performance drops a lot, it means that this part of the model contributes heavily to the models final performance. This is commonly done in 1 table, while disabling different parts of the model. Note that you can also do this the other way around, i.e. use only one feature (group) at a time, and test performance
* **Learning curve**: Evaluate how much data your model needs to reach a certain performance. Especially for the data augmentation projects this is essential.
* **Quantitative analysis**: Automated means of analyzing in which cases your model performs worse. This can for example be done with a confusion matrix.
* **Qualitative analysis**: Manually inspect a certain number of errors, and try to categorize them/find trends. Can be combined with the quantitative analysis, i.e., inspect 100 cases of positive reviews predicted to be negative and 100 cases of negative reviews predicted to be positive
* **Feature importance**: In traditional machine learning methods, one can often extract and inspect the weights of the features. In sklearn these can be found in: `trained_model.coef_`
* **Other metrics**: per class scores, partial matches, or count how often the span-borders were correct, but the label wrong.
* **Input words importance**: To gain insight into which words have a impact on prediction performance (positive, negative), we can analyze per-word impact: given a trained model, replace a given word with
the unknown word token and observe the change in prediction score (probability for a class). This is
shown in Figure 4 of [Rethmeier et al (2018)](https://aclweb.org/anthology/W18-6246) (a paper on controversy detection), also shown below: red-colored
tokens were important for controversy detection, blue-colored token decreased prediction scores.

<img width=400px src=example.png>

Note that this is a non-exhaustive list, and you are encouraged to also explore additional analyses.

### Checklist final project
Please check all these items before handing in your final report. You only have to upload a pdf file on learnit, and make sure a link to the code is included in the report and the code is accesible. 

* Are all group members and their email addresses specified?
* Does the group report include a representative project title?
* Does the group report contain an abstract?
* Does the introduction clearly specify the research intention and research question?
* Does the group report adequately refer to the relevant literature?
* Does the group report properly use figure, tables and examples?
* Does the group report provide and discuss the empirical results?
* Is the group report proofread?
* Does the pdf contain the link to the project’s github repo?
* Is the github repo accessible to the public (within ITU)?
* Is the group report maximum 5 pages long, excluding references and appendix?
* Are the group contributions added in the appendix?
* Does the repository contain all scripts and code to reproduce the results in the group report? Are instructions
 provided on how to run the code?
